# Exercício — Q-Learning vs. SARSA
### Aprendizado por Reforço | MBA em Ciência de Dados e IA

---

## Contexto

Você já aprendeu os fundamentos do Q-Learning e do SARSA. Ambos os algoritmos usam uma **Q-table** e a estratégia **ε-greedy**, mas diferem em **como atualizam** essa tabela:

| | Q-Learning | SARSA |
|---|---|---|
| Tipo | Off-policy | On-policy |
| Atualização usa... | `max Q(s', a')` — a **melhor** ação possível | `Q(s', a')` — a ação **de fato** escolhida |
| Comportamento | Otimista / mais agressivo | Conservador / mais seguro |

### Equações de atualização

**Q-Learning (off-policy):**
$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[ r + \gamma \cdot \underbrace{\max_{a'} Q(s', a')}_{\text{melhor ação possível}} - Q(s,a) \Big]$$

**SARSA (on-policy):**
$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[ r + \gamma \cdot \underbrace{Q(s', a')}_{\text{ação real tomada}} - Q(s,a) \Big]$$

---

## Ambiente — O Penhasco (*Cliff Walking*)

Este é o ambiente clássico de **Sutton & Barto (2018)** — o exemplo mais usado para demonstrar a diferença entre os dois algoritmos.

```
 ┌────────────────────────────────────────┐
 │  S  ·    ·    ·    ·    ·    ·    ·   G │  ← Linha 0 (topo)
 │  ·  ·    ·    ·    ·    ·    ·    ·   · │  ← Linha 1
 │  ·  ·    ·    ·    ·    ·    ·    ·   · │  ← Linha 2
 │  I  ☠    ☠    ☠    ☠    ☠    ☠    ☠   G │  ← Linha 3 (base)
 └────────────────────────────────────────┘
      col→  0    1    2    3    4    5    6    7    8
```

- **S / I** = Início (linha 3, coluna 0)
- **G** = Objetivo (linha 3, coluna 8)
- **☠** = Penhasco — recompensa **-100** e retorna ao início
- **·** = Célula livre — recompensa **-1** por passo

### Por que este ambiente é perfeito para comparar os dois algoritmos?

Existe um caminho **curto mas arriscado** (pela borda do penhasco) e um caminho **longo mas seguro** (pela parte de cima do grid).  
O Q-Learning tende a aprender o caminho curto (otimista).  
O SARSA tende a aprender o caminho seguro (conservador).

---

## Objetivos do Exercício

1. Implementar a **equação de atualização** do Q-Learning
2. Implementar a **equação de atualização** do SARSA
3. Comparar o comportamento e desempenho dos dois algoritmos
4. Interpretar os resultados e responder às perguntas de reflexão

---
## Célula 1 — Importações e Configuração

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Bibliotecas carregadas com sucesso!")

---
## Célula 2 — Definição do Ambiente

> **Leia com atenção.** Esta célula define o ambiente completo. Você não precisa modificá-la.

In [ ]:
# ─── Dimensões do grid ───────────────────────────────────────────────────────
ROWS = 4
COLS = 9

# ─── Posições especiais ───────────────────────────────────────────────────────
START = (3, 0)   # Canto inferior esquerdo
GOAL  = (3, 8)   # Canto inferior direito

# Penhasco: todas as células da base entre o início e o objetivo
CLIFF = [(3, c) for c in range(1, 8)]

# ─── Recompensas ──────────────────────────────────────────────────────────────
R_STEP  = -1    # Penalidade por cada passo (incentiva caminhos curtos)
R_CLIFF = -100  # Cair no penhasco — retorna ao início
R_GOAL  = 0     # Recompensa ao chegar (a penalidade de passo já está inclusa)

# ─── Ações ───────────────────────────────────────────────────────────────────
ACTIONS = [0, 1, 2, 3]
ACTION_NAMES  = {0: 'Cima', 1: 'Baixo', 2: 'Esquerda', 3: 'Direita'}
ACTION_ARROWS = {0: '↑',   1: '↓',    2: '←',         3: '→'}
ACTION_DELTAS = {0: (-1,0), 1: (1,0),  2: (0,-1),       3: (0,1)}


# ─── Função de transição do ambiente ─────────────────────────────────────────
def step(state, action):
    """
    Executa uma ação no ambiente.

    Parâmetros:
        state  : tupla (row, col) — estado atual
        action : int  0=Cima, 1=Baixo, 2=Esquerda, 3=Direita

    Retorna:
        next_state : tupla (row, col)
        reward     : float
        done       : bool — True se o episódio terminou
    """
    r, c = state
    dr, dc = ACTION_DELTAS[action]
    nr = np.clip(r + dr, 0, ROWS - 1)   # não sai pelos limites verticais
    nc = np.clip(c + dc, 0, COLS - 1)   # não sai pelos limites horizontais

    next_state = (nr, nc)

    # ── Chegou ao objetivo ──
    if next_state == GOAL:
        return GOAL, R_GOAL, True

    # ── Caiu no penhasco ──
    if next_state in CLIFF:
        return START, R_CLIFF, False   # reinicia episódio (mas não termina!)

    # ── Passo normal ──
    return next_state, R_STEP, False


# ─── Política ε-greedy ────────────────────────────────────────────────────────
def choose_action(state, Q, epsilon):
    """
    Seleciona uma ação usando a estratégia ε-greedy.
      - Com probabilidade ε   → ação aleatória  (exploração)
      - Com probabilidade 1-ε → argmax Q(s, ·)  (explotação)
    """
    if random.random() < epsilon:
        return random.choice(ACTIONS)
    r, c = state
    return int(np.argmax(Q[r, c]))


print(f"Ambiente configurado: grid {ROWS}×{COLS}")
print(f"  Início   : {START}")
print(f"  Objetivo : {GOAL}")
print(f"  Penhasco : {len(CLIFF)} células")
print()
print("Teste rápido a partir do estado inicial:")
for a in ACTIONS:
    ns, r, d = step(START, a)
    print(f"  {ACTION_ARROWS[a]} {ACTION_NAMES[a]:9s} → {ns}  recompensa={r:+4d}  fim={d}")

---
## Célula 3 — Visualização do Ambiente

In [ ]:
def plot_env(Q_ql=None, Q_sarsa=None, path_ql=None, path_sarsa=None, title="Cliff Walking"):
    """
    Visualiza o ambiente, e opcionalmente a política aprendida e o caminho percorrido.
    Aceita até dois agentes lado a lado: Q-Learning (esq.) e SARSA (dir.).
    """
    n_plots = 1 + (Q_sarsa is not None)
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 4))
    if n_plots == 1:
        axes = [axes]

    configs = []
    if Q_ql is not None or n_plots == 1:
        configs.append((Q_ql, path_ql, 'Q-Learning'))
    if Q_sarsa is not None:
        configs.append((Q_sarsa, path_sarsa, 'SARSA'))
    if not configs:
        configs = [(None, None, 'Ambiente')]

    COR_LIVRE    = '#F5F5F5'
    COR_PENHASCO = '#EF5350'
    COR_INICIO   = '#FFF9C4'
    COR_GOAL     = '#A5D6A7'
    COR_CAMINHO  = {'Q-Learning': '#1565C0', 'SARSA': '#6A1B9A'}

    for ax, (Q, path, label) in zip(axes, configs):
        for r in range(ROWS):
            for c in range(COLS):
                pos = (r, c)
                if pos in CLIFF:
                    cor = COR_PENHASCO
                elif pos == START:
                    cor = COR_INICIO
                elif pos == GOAL:
                    cor = COR_GOAL
                else:
                    cor = COR_LIVRE

                ax.add_patch(plt.Rectangle(
                    (c, ROWS - r - 1), 1, 1,
                    color=cor, zorder=1
                ))
                ax.add_patch(plt.Rectangle(
                    (c, ROWS - r - 1), 1, 1,
                    fill=False, edgecolor='#BDBDBD', linewidth=0.8, zorder=2
                ))

                # Ícone
                icon = ''
                if pos in CLIFF:  icon = '☠'
                if pos == START:  icon = '🚀'
                if pos == GOAL:   icon = '★'
                if icon:
                    ax.text(c + 0.5, ROWS - r - 0.35, icon,
                            ha='center', va='center', fontsize=14, zorder=4)

                # Política aprendida (seta + Q-value)
                if Q is not None and pos not in CLIFF and pos != GOAL:
                    best_a = int(np.argmax(Q[r, c]))
                    best_q = np.max(Q[r, c])
                    cor_seta = COR_CAMINHO.get(label, '#1565C0')
                    ax.text(c + 0.5, ROWS - r - 0.52, ACTION_ARROWS[best_a],
                            ha='center', va='center', fontsize=16,
                            color=cor_seta, fontweight='bold', zorder=3)
                    ax.text(c + 0.5, ROWS - r - 0.85, f'{best_q:.0f}',
                            ha='center', va='center', fontsize=7,
                            color='#757575', zorder=3)

        # Caminho
        if path:
            cor_linha = COR_CAMINHO.get(label, '#1565C0')
            xs = [p[1] + 0.5 for p in path]
            ys = [ROWS - p[0] - 0.5 for p in path]
            ax.plot(xs, ys, '-o', color=cor_linha, linewidth=2.5,
                    markersize=5, zorder=5, alpha=0.8)

        ax.set_xlim(0, COLS)
        ax.set_ylim(0, ROWS)
        ax.set_xticks(np.arange(COLS) + 0.5)
        ax.set_yticks(np.arange(ROWS) + 0.5)
        ax.set_xticklabels(range(COLS), fontsize=9)
        ax.set_yticklabels(range(ROWS - 1, -1, -1), fontsize=9)
        ax.set_title(label, fontsize=12, fontweight='bold', pad=8)

    legend_els = [
        mpatches.Patch(color=COR_PENHASCO, label='Penhasco (−100)'),
        mpatches.Patch(color=COR_GOAL,    label='Objetivo'),
        mpatches.Patch(color=COR_INICIO,  label='Início 🚀'),
    ]
    axes[0].legend(handles=legend_els, loc='upper left', fontsize=8, framealpha=0.95)

    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


# Visualiza o ambiente vazio antes do treinamento
plot_env(title='Ambiente — Cliff Walking (antes do treinamento)')

---
## Exercício 1 — Implementar o Q-Learning

Preencha os `# TODO` abaixo para implementar o loop de treinamento do Q-Learning.

**Lembre-se da equação:**
$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[ r + \gamma \cdot \max_{a'} Q(s', a') - Q(s,a) \Big]$$

> **Dica:** A diferença entre Q-Learning e SARSA está em **uma única linha** do código de atualização. No Q-Learning, o alvo usa `np.max(Q[nr, nc])`. No SARSA, usará o Q-value da ação já escolhida.

In [ ]:
def treinar_qlearning(alpha, gamma, epsilon, n_episodes, max_steps=500):
    """
    Treina um agente com Q-Learning.

    Parâmetros:
        alpha      : float — taxa de aprendizado  (0 < α ≤ 1)
        gamma      : float — fator de desconto    (0 < γ ≤ 1)
        epsilon    : float — probabilidade de exploração
        n_episodes : int   — número de episódios de treinamento
        max_steps  : int   — limite de passos por episódio

    Retorna:
        Q             : np.ndarray (ROWS, COLS, 4) — Q-table final
        hist_reward   : list[float] — recompensa total por episódio
    """
    # ── Inicializa a Q-table com zeros ────────────────────────────────────────
    Q = np.zeros((ROWS, COLS, len(ACTIONS)))
    hist_reward = []

    for ep in range(n_episodes):
        state        = START
        total_reward = 0

        for _ in range(max_steps):
            # ── Passo 1: escolher ação com política ε-greedy ──────────────────
            action = choose_action(state, Q, epsilon)

            # ── Passo 2: interagir com o ambiente ─────────────────────────────
            next_state, reward, done = step(state, action)

            # ── Passo 3: atualizar a Q-table (equação de Bellman) ─────────────
            r,  c  = state
            nr, nc = next_state

            # TODO: calcule o 'alvo' do Q-Learning
            # Lembre-se: usa o MÁXIMO dos Q-values do próximo estado
            alvo = None  # ← substitua None pela expressão correta

            # TODO: aplique a equação de atualização
            # Q[r, c, action] += ...

            total_reward += reward
            state         = next_state

            if done:
                break

        hist_reward.append(total_reward)

    return Q, hist_reward


# ─── Hiperparâmetros ──────────────────────────────────────────────────────────
ALPHA     = 0.1
GAMMA     = 0.99
EPSILON   = 0.1
N_EPISODES = 500

# ─── Treinamento ─────────────────────────────────────────────────────────────
Q_ql, hist_ql = treinar_qlearning(ALPHA, GAMMA, EPSILON, N_EPISODES)

print(f"Q-Learning — {N_EPISODES} episódios concluídos.")
print(f"Recompensa média (últimos 50 ep.): {np.mean(hist_ql[-50:]):.1f}")

---
## Exercício 2 — Implementar o SARSA

Agora implemente o SARSA. A estrutura é quase idêntica à do Q-Learning —  
mas preste atenção na diferença crucial da equação de atualização.

**Lembre-se da equação:**
$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[ r + \gamma \cdot Q(s', a') - Q(s,a) \Big]$$

> **Dica 1:** No SARSA, você precisa escolher a próxima ação (`next_action`) **antes** de atualizar a Q-table.  
> **Dica 2:** A ação que entra na equação é exatamente aquela que será executada no **próximo passo** do loop.

In [ ]:
def treinar_sarsa(alpha, gamma, epsilon, n_episodes, max_steps=500):
    """
    Treina um agente com SARSA.

    Parâmetros:
        alpha      : float — taxa de aprendizado
        gamma      : float — fator de desconto
        epsilon    : float — probabilidade de exploração
        n_episodes : int   — número de episódios
        max_steps  : int   — limite de passos por episódio

    Retorna:
        Q           : np.ndarray (ROWS, COLS, 4) — Q-table final
        hist_reward : list[float] — recompensa total por episódio
    """
    Q = np.zeros((ROWS, COLS, len(ACTIONS)))
    hist_reward = []

    for ep in range(n_episodes):
        state  = START
        total_reward = 0

        # ── SARSA começa escolhendo a ação ANTES do loop ──────────────────────
        # TODO: escolha a ação inicial usando choose_action
        action = None  # ← substitua None pela chamada correta

        for _ in range(max_steps):
            # ── Passo 1: interagir com o ambiente ─────────────────────────────
            next_state, reward, done = step(state, action)

            # ── Passo 2: escolher a PRÓXIMA ação (ainda com ε-greedy) ──────────
            # TODO: escolha next_action com choose_action
            next_action = None  # ← substitua None

            # ── Passo 3: atualizar a Q-table (equação SARSA) ──────────────────
            r,  c  = state
            nr, nc = next_state

            # TODO: calcule o 'alvo' do SARSA
            # Atenção: usa Q[nr, nc, next_action], NÃO np.max
            alvo = None  # ← substitua None

            # TODO: aplique a equação de atualização
            # Q[r, c, action] += ...

            total_reward += reward

            # ── Passo 4: avança para o próximo estado/ação ────────────────────
            # TODO: atualize state e action para o próximo passo
            state  = None  # ← substitua
            action = None  # ← substitua

            if done:
                break

        hist_reward.append(total_reward)

    return Q, hist_reward


# ─── Treinamento (mesmos hiperparâmetros para comparação justa) ───────────────
Q_sarsa, hist_sarsa = treinar_sarsa(ALPHA, GAMMA, EPSILON, N_EPISODES)

print(f"SARSA — {N_EPISODES} episódios concluídos.")
print(f"Recompensa média (últimos 50 ep.): {np.mean(hist_sarsa[-50:]):.1f}")

---
## Exercício 3 — Comparar as Curvas de Aprendizado

Execute a célula abaixo para visualizar a evolução dos dois algoritmos lado a lado.

In [ ]:
def media_movel(dados, janela=20):
    return np.convolve(dados, np.ones(janela) / janela, mode='valid')

JANELA = 20
x = range(JANELA - 1, N_EPISODES)

fig, ax = plt.subplots(figsize=(11, 4))

# Curvas brutas (transparentes)
ax.plot(hist_ql,    alpha=0.15, color='#1565C0', linewidth=0.8)
ax.plot(hist_sarsa, alpha=0.15, color='#6A1B9A', linewidth=0.8)

# Médias móveis
ax.plot(x, media_movel(hist_ql,    JANELA), color='#1565C0', linewidth=2.5,
        label=f'Q-Learning  (média {JANELA} ep.)')
ax.plot(x, media_movel(hist_sarsa, JANELA), color='#6A1B9A', linewidth=2.5,
        label=f'SARSA       (média {JANELA} ep.)')

ax.axhline(y=np.mean(hist_ql[-50:]),    color='#1565C0', linestyle='--',
           alpha=0.5, label=f'QL média final: {np.mean(hist_ql[-50:]):.1f}')
ax.axhline(y=np.mean(hist_sarsa[-50:]), color='#6A1B9A', linestyle='--',
           alpha=0.5, label=f'SARSA média final: {np.mean(hist_sarsa[-50:]):.1f}')

ax.set_title('Curva de Aprendizado — Q-Learning vs. SARSA',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Episódio', fontsize=11)
ax.set_ylabel('Recompensa total por episódio', fontsize=11)
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nResumo:")
print(f"  Q-Learning — recompensa média (últimos 50 ep.): {np.mean(hist_ql[-50:]):>7.1f}")
print(f"  SARSA      — recompensa média (últimos 50 ep.): {np.mean(hist_sarsa[-50:]):>7.1f}")
print()
print("  Qual obteve maior recompensa durante o TREINAMENTO? Qual esperaria para EXECUÇÃO?")

---
## Exercício 4 — Política Aprendida e Caminho

A célula abaixo extrai o caminho **greedy** (sem exploração) de cada agente e visualiza a política aprendida.

In [ ]:
def extrair_caminho(Q, max_steps=100):
    """
    Executa o agente de forma greedy (ε=0) e retorna o caminho percorrido.
    """
    state    = START
    caminho  = [state]
    visitados = {state}

    for _ in range(max_steps):
        r, c   = state
        action = int(np.argmax(Q[r, c]))
        next_state, _, done = step(state, action)

        # Interrompe em loop
        if next_state in visitados and next_state != GOAL:
            print("  [Aviso] Loop detectado — agente pode não ter convergido.")
            break

        caminho.append(next_state)
        visitados.add(next_state)
        state = next_state

        if done:
            break

    return caminho


path_ql    = extrair_caminho(Q_ql)
path_sarsa = extrair_caminho(Q_sarsa)

chegou_ql    = path_ql[-1] == GOAL
chegou_sarsa = path_sarsa[-1] == GOAL

print(f"Q-Learning : {'✅ chegou' if chegou_ql    else '❌ não chegou'} em {len(path_ql)-1} passos")
print(f"SARSA      : {'✅ chegou' if chegou_sarsa  else '❌ não chegou'} em {len(path_sarsa)-1} passos")
print()

# Visualiza os dois agentes lado a lado
plot_env(
    Q_ql=Q_ql,       Q_sarsa=Q_sarsa,
    path_ql=path_ql, path_sarsa=path_sarsa,
    title='Política e Caminho Aprendido — Q-Learning vs. SARSA'
)

---
## Exercício 5 — Impacto de ε na Diferença entre os Algoritmos

O valor de **ε** é a peça-chave para entender **por que SARSA e Q-Learning se comportam diferente** neste ambiente.

- Quando **ε = 0** (sem exploração), os dois algoritmos convergem para a **mesma política** (o caminho mais curto).
- Quando **ε > 0**, o SARSA *sabe* que cometerá erros aleatórios e aprende a se afastar do penhasco.  
  O Q-Learning ignora isso — planeja para o melhor caso e sofre as consequências durante a exploração.

Execute a célula abaixo para comparar os dois algoritmos com três valores de ε.

In [ ]:
epsilons_teste = [0.0, 0.1, 0.3]

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
fig.suptitle('Impacto de ε na Recompensa Final (últimos 50 episódios)',
             fontsize=13, fontweight='bold')

for ax, eps in zip(axes, epsilons_teste):
    np.random.seed(SEED); random.seed(SEED)
    _, h_ql = treinar_qlearning(ALPHA, GAMMA, eps, N_EPISODES)

    np.random.seed(SEED); random.seed(SEED)
    _, h_sa = treinar_sarsa(ALPHA, GAMMA, eps, N_EPISODES)

    x = range(JANELA - 1, N_EPISODES)
    ax.plot(x, media_movel(h_ql, JANELA), color='#1565C0', linewidth=2, label='Q-Learning')
    ax.plot(x, media_movel(h_sa, JANELA), color='#6A1B9A', linewidth=2, label='SARSA')

    media_ql = np.mean(h_ql[-50:])
    media_sa = np.mean(h_sa[-50:])

    ax.set_title(f'ε = {eps}\nQL: {media_ql:.1f}  |  SARSA: {media_sa:.1f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Episódio', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Recompensa média', fontsize=10)
plt.tight_layout()
plt.show()

---
## Perguntas de Reflexão

Responda às perguntas abaixo na célula de texto seguinte. Use os gráficos e resultados obtidos para embasar suas respostas.

---

**1.** Observe o gráfico do Exercício 3. Durante o treinamento (com ε > 0), qual algoritmo obteve maior recompensa média — Q-Learning ou SARSA? Por quê? Isso é esperado pela teoria?

**2.** Observe o Exercício 4. O caminho do Q-Learning e do SARSA são iguais ou diferentes? Explique, em termos das equações de atualização, por que isso acontece neste ambiente específico.

**3.** Observe o Exercício 5. O que acontece com a diferença entre Q-Learning e SARSA à medida que ε se aproxima de zero? Que conclusão você tira sobre a relação entre ε e o comportamento on-policy vs. off-policy?

**4.** Em qual cenário de negócios você preferiria usar SARSA em vez de Q-Learning? Justifique com base no conceito de on-policy vs. off-policy e no trade-off entre exploração e segurança.

**5. (Desafio)** O Q-Learning é chamado de **off-policy** porque aprende a política ótima independente do que o agente executa. Se você remover completamente a exploração (ε = 0) após o treinamento e simular os dois agentes, os resultados convergem? Execute um experimento e explique o resultado.

### Suas Respostas

**Questão 1:**  
*(escreva aqui)*

**Questão 2:**  
*(escreva aqui)*

**Questão 3:**  
*(escreva aqui)*

**Questão 4:**  
*(escreva aqui)*

**Questão 5 (Desafio):**  
*(escreva aqui)*

---
## Resumo — O que você praticou neste exercício

| Habilidade | O que foi exercitado |
|---|---|
| Implementação | Equação de Bellman para Q-Learning e SARSA |
| Conceito central | Diferença on-policy vs. off-policy |
| Análise crítica | Leitura e interpretação de curvas de aprendizado |
| Pensamento aplicado | Trade-off entre risco e otimidade em negócios |
| Experimentação | Impacto de hiperparâmetros (ε) no comportamento dos agentes |

### Próximo passo natural

A Q-table funciona bem para espaços de estados discretos e pequenos.  
Quando o espaço de estados explode (pixels de vídeo, variáveis contínuas), a solução é substituir a tabela por uma **rede neural** — esse é o **Deep Q-Network (DQN)**, base dos agentes que jogam Atari e jogos complexos.